In [1]:
!pip install encodec
import torch, torchaudio, librosa, json
import numpy as np
import pandas as pd
from pathlib import Path

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 33.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for encodec: filename=encodec-0.1.1-py3-none-any.whl size=45851 sha256=3a7ebaba2c2d6787400262dfbd5624c1f8d3a7901620811c75a03230fae61165
  Stored in directory: /root/.cache/pip/wheels/b8/eb/9f/e13610cc46ab39d3199fbabebd1c3e142d44b679526e0f228a
Successfully built encodec


In [2]:
import random
random.seed(42)

def quick_rms(file_path):
    try:
        wav, sr = torchaudio.load(file_path)
        if wav.shape[0] > 1: wav = wav.mean(dim=0, keepdim=True)
        if sr != 8000: wav = torchaudio.functional.resample(wav, sr, 8000)
        return float(wav.pow(2).mean().sqrt())
    except Exception:
        return 0.0

def stratified_sample(all_files, num_samples, low_ratio=0.5):
    scored = [(quick_rms(f), f) for f in all_files]
    scored.sort(key=lambda x: x[0])
    mid = len(scored) // 2
    low_pool, high_pool = [f for _, f in scored[:mid]], [f for _, f in scored[mid:]]
    n_low = min(int(num_samples * low_ratio), len(low_pool))
    n_high = min(num_samples - n_low, len(high_pool))
    sampled = random.sample(low_pool, n_low) + random.sample(high_pool, n_high)
    random.shuffle(sampled)
    return sampled

SLAKH_BASE = Path("/kaggle/input/datasets/alonhaviv/slakh2100/slakh2100_flac_redux")
def get_all_mixes(split_name):
    return sorted(list((SLAKH_BASE / split_name).rglob("mix.flac")))

train_mixes = stratified_sample(get_all_mixes("train"), 800, low_ratio=0.6)
test_mixes  = stratified_sample(get_all_mixes("test"), 200, low_ratio=0.6)
print(f"Reproduced {len(train_mixes)} train + {len(test_mixes)} test paths")

Reproduced 800 train + 151 test paths


In [3]:
def collect_rms_samples(file_paths, n_samples_per_track=50):
    all_rms = []
    for fp in file_paths:
        try:
            wav, sr = torchaudio.load(fp)
            if wav.shape[0] > 1:
                wav = wav.mean(dim=0, keepdim=True)
            frame_size = sr // 50
            n_frames = wav.shape[-1] // frame_size
            if n_frames == 0:
                continue
            rms = wav[0, :n_frames*frame_size].unfold(0, frame_size, frame_size).pow(2).mean(-1).sqrt()
            idx = torch.randperm(len(rms))[:n_samples_per_track]
            all_rms.extend(rms[idx].tolist())
        except Exception:
            continue
    return all_rms

SKYRIM_BASE = Path("/kaggle/input/datasets/danielebracoloni/skyrim-ost")
WITCHER_BASE = Path("/kaggle/input/datasets/danielebracoloni/witcher3-ost")

ost_file_paths = (sorted(SKYRIM_BASE.rglob("*.mp3")) + sorted(SKYRIM_BASE.rglob("*.flac")) +
                   sorted(WITCHER_BASE.rglob("*.mp3")) + sorted(WITCHER_BASE.rglob("*.flac")))
print(f"Found {len(ost_file_paths)} OST audio files")

slakh_rms = collect_rms_samples(train_mixes[:200] + test_mixes[:50])
ost_rms   = collect_rms_samples(ost_file_paths)

p_low_s, p_high_s = np.percentile(slakh_rms, [5, 95])
p_low_o, p_high_o = np.percentile(ost_rms, [5, 95])

tension_stats = {
    "p_low":  float(np.mean([p_low_s, p_low_o])),
    "p_high": float(np.mean([p_high_s, p_high_o])),
}
print("tension_rms:", tension_stats)

Found 74 OST audio files
tension_rms: {'p_low': 0.00793589388485998, 'p_high': 0.21743767708539957}


In [4]:
raw_cols = ["track_id", "perc_ratio", "percussive_loudness",
            "note_shortness", "dissonance", "spectral_centroid_mean"]

df_slakh = pd.read_csv("/kaggle/input/datasets/danielebracoloni/slakh-and-custom-csvs/slakh_raw_features.csv")
df_skyrim_raw = pd.read_csv("/kaggle/input/datasets/danielebracoloni/slakh-and-custom-csvs/combat_chill_profile_normalized_v2.csv")[raw_cols]
df_witcher_raw = pd.read_csv("/kaggle/input/datasets/danielebracoloni/slakh-and-custom-csvs/combat_chill_profile_W3_raw.csv")[raw_cols]

df_ost = pd.concat([df_skyrim_raw, df_witcher_raw], ignore_index=True)
print(f"Slakh: {len(df_slakh)} tracks, OST (Skyrim+Witcher3): {len(df_ost)} tracks")

features = ["perc_ratio", "percussive_loudness", "note_shortness", "dissonance", "spectral_centroid_mean"]
norm_stats = {}
for feat in features:
    p_low_slakh, p_high_slakh = np.percentile(df_slakh[feat], [5, 95])
    p_low_ost,   p_high_ost   = np.percentile(df_ost[feat],   [5, 95])
    norm_stats[feat] = {
        "p_low":  float(np.mean([p_low_slakh, p_low_ost])),
        "p_high": float(np.mean([p_high_slakh, p_high_ost])),
    }
    print(f"{feat}: Slakh=({p_low_slakh:.3f},{p_high_slakh:.3f})  "
          f"OST=({p_low_ost:.3f},{p_high_ost:.3f})  Balanced=({norm_stats[feat]['p_low']:.3f},{norm_stats[feat]['p_high']:.3f})")

norm_stats["tension_rms"] = tension_stats  # from Cell 3

with open("norm_stats.json", "w") as f:
    json.dump(norm_stats, f, indent=2)

print("Saved complete norm_stats.json:")
print(json.dumps(norm_stats, indent=2))

Slakh: 951 tracks, OST (Skyrim+Witcher3): 75 tracks
perc_ratio: Slakh=(0.087,0.346)  OST=(0.010,0.204)  Balanced=(0.049,0.275)
percussive_loudness: Slakh=(0.024,0.057)  OST=(0.006,0.068)  Balanced=(0.015,0.063)
note_shortness: Slakh=(0.594,0.857)  OST=(0.000,0.889)  Balanced=(0.297,0.873)
dissonance: Slakh=(0.246,0.487)  OST=(0.203,0.513)  Balanced=(0.224,0.500)
spectral_centroid_mean: Slakh=(1053.425,2007.778)  OST=(738.168,2326.144)  Balanced=(895.797,2166.961)
Saved complete norm_stats.json:
{
  "perc_ratio": {
    "p_low": 0.04851383623301943,
    "p_high": 0.2747280395
  },
  "percussive_loudness": {
    "p_low": 0.014788103522732776,
    "p_high": 0.06263642851263282
  },
  "note_shortness": {
    "p_low": 0.29704859592444993,
    "p_high": 0.8731559347706688
  },
  "dissonance": {
    "p_low": 0.22418186292052267,
    "p_high": 0.5004020690917969
  },
  "spectral_centroid_mean": {
    "p_low": 895.796685121579,
    "p_high": 2166.9607449151263
  },
  "tension_rms": {
    "p_low"

Tokenizer


In [5]:
import random
import shutil
import json
import traceback
from pathlib import Path

import torch
import torchaudio
import librosa
import numpy as np

from encodec import EncodecModel
from encodec.utils import convert_audio

# ── Reproducibility ──
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# EnCodec 24 kHz uses a hop length of 320 samples:
# 24,000 / 320 = 75 codec frames per second.
FRAME_RATE = 75
BANDWIDTH_KBPS = 24.0

# ── Output directories ──
OUTPUT_DIR = Path("/kaggle/working/tensors_final")
TRAIN_OUT = OUTPUT_DIR / "train"
TEST_OUT = OUTPUT_DIR / "test"
TRAIN_OUT.mkdir(parents=True, exist_ok=True)
TEST_OUT.mkdir(parents=True, exist_ok=True)

# ── Resume support from previous output ──
PREVIOUS_OUTPUT = Path(
    "/kaggle/input/datasets/danielebracoloni/"
    "tesors-slakh-375-checkpoint/tensors_final"
)

assert PREVIOUS_OUTPUT.exists(), (
    f"Checkpoint path does not exist: {PREVIOUS_OUTPUT}"
)

print("Checkpoint contents:")
print("train:", len(list((PREVIOUS_OUTPUT / "train").glob("*.pt"))))
print("test:", len(list((PREVIOUS_OUTPUT / "test").glob("*.pt"))))

if PREVIOUS_OUTPUT.exists():
    for split in ["train", "test"]:
        src_dir = PREVIOUS_OUTPUT / split
        dst_dir = OUTPUT_DIR / split

        if src_dir.exists():
            for source_file in src_dir.glob("*.pt"):
                destination_file = dst_dir / source_file.name
                if not destination_file.exists():
                    shutil.copy2(source_file, destination_file)

    n_existing = (
        len(list(TRAIN_OUT.glob("*.pt"))) +
        len(list(TEST_OUT.glob("*.pt")))
    )
    print(f"Resumed from previous output: {n_existing} tracks already present.")
else:
    print("No previous partial output found — starting fresh.")

# ── Load shared normalization statistics ──
with open("norm_stats.json", "r") as f:
    NORM_STATS = json.load(f)

required_stats = {
    "perc_ratio",
    "percussive_loudness",
    "note_shortness",
    "dissonance",
    "spectral_centroid_mean",
    "tension_rms",
}
missing_stats = required_stats - set(NORM_STATS)
if missing_stats:
    raise KeyError(f"norm_stats.json is missing: {sorted(missing_stats)}")

def normalize(value, feature_name):
    stats = NORM_STATS[feature_name]
    return float(np.clip(
        (value - stats["p_low"]) /
        (stats["p_high"] - stats["p_low"] + 1e-8),
        0.0,
        1.0,
    ))

# ── EnCodec setup ──
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = EncodecModel.encodec_model_24khz()
model.set_target_bandwidth(BANDWIDTH_KBPS)
model.to(device)
model.eval()

print(
    f"EnCodec: {model.sample_rate} Hz, "
    f"{model.channels} channel(s), "
    f"{BANDWIDTH_KBPS} kbps, expected frame rate: {FRAME_RATE} Hz"
)

# ── Causal rolling-window combat features ──
def compute_combat_features_from_array(y, sr):
    y_harmonic, y_percussive = librosa.effects.hpss(y)

    harmonic_energy = np.sum(y_harmonic ** 2)
    percussive_energy = np.sum(y_percussive ** 2)
    perc_ratio = percussive_energy / (
        harmonic_energy + percussive_energy + 1e-8
    )

    perc_loudness = float(np.sqrt(np.mean(y_percussive ** 2)))

    onsets = librosa.onset.onset_detect(y=y, sr=sr, units="time")
    ioi = float(np.mean(np.diff(onsets))) if len(onsets) >= 2 else 1.5
    note_shortness = float(np.clip(1.0 - ioi / 1.5, 0.0, 1.0))

    chroma = librosa.feature.chroma_cqt(y=y, sr=sr)
    adjacent_energy = np.sum(chroma[:-1] * chroma[1:], axis=0)
    dissonance = float(np.mean(
        adjacent_energy / (np.sum(chroma, axis=0) + 1e-8)
    ))

    centroid = float(np.mean(
        librosa.feature.spectral_centroid(y=y, sr=sr)
    ))

    return perc_ratio, perc_loudness, note_shortness, dissonance, centroid


def compute_causal_combat_score_track(
    y,
    sr,
    window_sec=5.0,
    hop_sec=1.0,
):
    """
    Each score at time t uses only audio in [t - 5 seconds, t].
    No centered/future window leakage.
    """
    hop_len = int(hop_sec * sr)
    window_len = int(window_sec * sr)
    n_hops = max(1, int(np.ceil(len(y) / hop_len)))

    scores = []

    for i in range(n_hops):
        end = min((i + 1) * hop_len, len(y))
        start = max(0, end - window_len)
        y_window = y[start:end]

        if len(y_window) < int(0.5 * sr):
            scores.append(scores[-1] if scores else 0.3)
            continue

        perc_ratio, perc_loudness, note_shortness, dissonance, centroid = (
            compute_combat_features_from_array(y_window, sr)
        )

        score = (
            0.30 * normalize(perc_ratio, "perc_ratio") +
            0.25 * normalize(perc_loudness, "percussive_loudness") +
            0.10 * normalize(note_shortness, "note_shortness") +
            0.15 * normalize(dissonance, "dissonance") +
            0.20 * normalize(centroid, "spectral_centroid_mean")
        )
        scores.append(score)

    return np.asarray(scores, dtype=np.float32), hop_sec


def align_causal_to_frames(window_scores, hop_sec, frame_rate, total_frames):
    """
    Causal sample-and-hold alignment:
    one rolling score lasts until the next score becomes available.
    """
    hop_frames = max(1, round(hop_sec * frame_rate))
    aligned = np.empty(total_frames, dtype=np.float32)

    for i, score in enumerate(window_scores):
        start = i * hop_frames
        end = min(start + hop_frames, total_frames)

        if start >= total_frames:
            break

        aligned[start:end] = score

    filled_until = min(len(window_scores) * hop_frames, total_frames)
    if filled_until < total_frames:
        aligned[filled_until:] = window_scores[-1]

    return aligned


# ── Slakh file discovery / reproducible stratification ──
SLAKH_BASE = Path(
    "/kaggle/input/datasets/alonhaviv/slakh2100/slakh2100_flac_redux"
)

def get_all_mixes(split_name):
    split_dir = SLAKH_BASE / split_name
    mix_files = sorted(split_dir.rglob("mix.flac"))

    if not mix_files:
        raise FileNotFoundError(f"No mix.flac files found under {split_dir}")

    return mix_files


def quick_rms(file_path):
    try:
        waveform, sample_rate = torchaudio.load(file_path)

        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        if sample_rate != 8000:
            waveform = torchaudio.functional.resample(
                waveform, sample_rate, 8000
            )

        return float(waveform.pow(2).mean().sqrt())
    except Exception:
        return 0.0


def stratified_sample(all_files, num_samples, low_ratio=0.5):
    scored = [(quick_rms(file_path), file_path) for file_path in all_files]
    scored.sort(key=lambda pair: pair[0])

    middle = len(scored) // 2
    low_pool = [file_path for _, file_path in scored[:middle]]
    high_pool = [file_path for _, file_path in scored[middle:]]

    n_low = min(int(num_samples * low_ratio), len(low_pool))
    n_high = min(num_samples - n_low, len(high_pool))

    sampled = (
        random.sample(low_pool, n_low) +
        random.sample(high_pool, n_high)
    )
    random.shuffle(sampled)

    return sampled


# ── One-track encoder ──
def process_track(file_path, save_name):
    save_dir = TRAIN_OUT if save_name.startswith("train") else TEST_OUT
    save_path = save_dir / f"{save_name}.pt"

    # Checkpoint / resume behavior
    if save_path.exists():
        return True

    try:
        track_id = file_path.parent.name

        waveform, source_sample_rate = torchaudio.load(file_path)
        original_samples = waveform.shape[-1]
        duration_sec = original_samples / source_sample_rate

        # Explicit mono conversion before EnCodec
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        waveform = convert_audio(
            waveform,
            source_sample_rate,
            model.sample_rate,
            model.channels,
        )

        peak = waveform.abs().max()
        if peak > 0:
            waveform = waveform / peak * 0.95

        # waveform is [channels, samples], normally [1, N].
        # EnCodec expects [batch, channels, samples].
        waveform_gpu = waveform.unsqueeze(0).to(device)

        with torch.no_grad():
            encoded_frames = model.encode(waveform_gpu)
            codes, scales = encoded_frames[0]
            tokens = codes[0].cpu()  # [n_q, T]

        # CRITICAL: waveform is already [1, N].
        # waveform[0] is [N]. Do NOT mean(dim=0), which would collapse N samples to one scalar.
        wav_mono = waveform[0].cpu()  # [N]

        num_samples = wav_mono.shape[-1]
        num_frames = tokens.shape[-1]

        if num_frames <= 0:
            raise ValueError("EnCodec returned zero frames.")

        # EnCodec 24 kHz uses 320 samples/frame = 75 Hz.
        # Derive the actual integer frame length safely from the data.
        frame_size = max(1, round(num_samples / num_frames))

        required_samples = num_frames * frame_size
        if num_samples < required_samples:
            wav_for_rms = torch.nn.functional.pad(
                wav_mono,
                (0, required_samples - num_samples),
            )
        else:
            wav_for_rms = wav_mono[:required_samples]

        rms_frames = torch.sqrt(
            torch.mean(
                wav_for_rms.reshape(num_frames, frame_size) ** 2,
                dim=-1,
            )
        )

        tension = np.clip(
            (
                rms_frames.numpy() -
                NORM_STATS["tension_rms"]["p_low"]
            ) /
            (
                NORM_STATS["tension_rms"]["p_high"] -
                NORM_STATS["tension_rms"]["p_low"] +
                1e-8
            ),
            0.0,
            1.0,
        ).astype(np.float32)

        window_scores, hop_sec = compute_causal_combat_score_track(
            wav_mono.numpy(),
            model.sample_rate,
        )

        combat_score = align_causal_to_frames(
            window_scores=window_scores,
            hop_sec=hop_sec,
            frame_rate=FRAME_RATE,
            total_frames=num_frames,
        )

        min_length = min(
            tokens.shape[-1],
            len(tension),
            len(combat_score),
        )

        tokens = tokens[:, :min_length]
        tension = torch.tensor(tension[:min_length], dtype=torch.float32)
        combat_score = torch.tensor(
            combat_score[:min_length],
            dtype=torch.float32,
        )

        if tokens.shape[0] != 32:
            print(
                f"Warning: {track_id} has {tokens.shape[0]} codebooks "
                f"at {BANDWIDTH_KBPS} kbps."
            )

        torch.save(
            {
                "track_id": track_id,
                "duration_sec": duration_sec,
                "original_samples": original_samples,
                "tokens": tokens,                  # [n_q, T]
                "tension": tension,                # [T]
                "combat_score": combat_score,      # [T]
                "sample_rate": model.sample_rate,
                "frame_rate": FRAME_RATE,
                "bandwidth_kbps": BANDWIDTH_KBPS,
            },
            save_path,
        )

        del waveform, waveform_gpu, wav_mono, wav_for_rms
        del tokens, tension, combat_score, encoded_frames, codes, scales
        torch.cuda.empty_cache()

        return True

    except Exception as error:
        print(f"\nFailed to process {file_path}: {error}")
        traceback.print_exc()
        torch.cuda.empty_cache()
        return False


# ── Report existing files after resume ──
existing_train_files = sorted(TRAIN_OUT.glob("train_*.pt"))
existing_test_files = sorted(TEST_OUT.glob("test_*.pt"))

print(f"Resumed train files: {len(existing_train_files)}")
print(f"Resumed test files: {len(existing_test_files)}")

if len(existing_train_files) == 0 and len(existing_test_files) == 0:
    print("WARNING: No existing .pt files found in OUTPUT_DIR. Starting from scratch.")

# ── Build exclusion set from track_id metadata already saved on disk ──
# This avoids any reliance on reproducing prior random.sample() calls exactly.
used_track_ids = set()
for pt_path in existing_train_files:
    data = torch.load(pt_path, map_location="cpu")
    used_track_ids.add(data["track_id"])

print(f"Already-used track_ids on disk: {len(used_track_ids)}")

# ── Encode 325 NEW tracks from Slakh's "train" split, excluding used ones ──
NUM_NEW = 325
LOW_RATIO = 0.6

all_train_mixes = get_all_mixes("train")
unused_train_mixes = [
    p for p in all_train_mixes
    if p.parent.name not in used_track_ids
]

print(f"Unused Slakh train tracks available: {len(unused_train_mixes)}")

new_mixes = stratified_sample(
    unused_train_mixes,
    NUM_NEW,
    low_ratio=LOW_RATIO,
)
if len(new_mixes) != NUM_NEW:
    raise ValueError(
        f"Expected {NUM_NEW} new tracks, got {len(new_mixes)}."
    )

new_track_ids = {track_path.parent.name for track_path in new_mixes}

overlap = used_track_ids & new_track_ids
if overlap:
    raise ValueError(
        f"Found {len(overlap)} overlapping track IDs: "
        f"{sorted(overlap)[:10]}"
    )

print(f"Verified {len(new_mixes)} unique unused Slakh tracks.")

print(f"Encoding {len(new_mixes)} new tracks from Slakh train split (unused).")

new_ok = 0
next_index_start = len(existing_train_files)  # continue numbering after last existing file

existing_train_indices = sorted(
    int(path.stem.split("_")[-1])
    for path in existing_train_files
)

expected_train_indices = list(range(len(existing_train_files)))

if existing_train_indices != expected_train_indices:
    raise ValueError(
        "Existing train files are not consecutively numbered from train_000."
    )

print(
    f"New files will be numbered "
    f"train_{next_index_start:03d} through "
    f"train_{next_index_start + NUM_NEW - 1:03d}."
)

for offset, track_path in enumerate(new_mixes):
    index = next_index_start + offset
    save_name = f"train_{index:03d}"
    if process_track(track_path, save_name):
        new_ok += 1
    if (offset + 1) % 25 == 0:
        print(f"New progress: {offset + 1}/{len(new_mixes)} | successful: {new_ok}")

print(f"New done: {new_ok}/{len(new_mixes)}")
# ── Final zip ──
shutil.make_archive(
    "/kaggle/working/slakh2100-encodec24k-32cb-combatscore-1395tracks",
    "zip",
    OUTPUT_DIR,
)

print("Artifact zipped.")

Checkpoint contents:
train: 1070
test: 151
Resumed from previous output: 1221 tracks already present.
Using device: cuda
Downloading: "https://dl.fbaipublicfiles.com/encodec/v0/encodec_24khz-d7cc33bc.th" to /root/.cache/torch/hub/checkpoints/encodec_24khz-d7cc33bc.th


/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
100%|██████████| 88.9M/88.9M [00:00<00:00, 178MB/s]


EnCodec: 24000 Hz, 1 channel(s), 24.0 kbps, expected frame rate: 75 Hz
Resumed train files: 1070
Resumed test files: 151
Already-used track_ids on disk: 1070
Unused Slakh train tracks available: 489
Verified 325 unique unused Slakh tracks.
Encoding 325 new tracks from Slakh train split (unused).
New files will be numbered train_1070 through train_1394.


/usr/local/lib/python3.12/dist-packages/librosa/core/pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(
/usr/local/lib/python3.12/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=1024 is too large for input signal of length=750
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=1024 is too large for input signal of length=375
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/librosa/core/pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(


New progress: 25/325 | successful: 25
New progress: 50/325 | successful: 50
New progress: 75/325 | successful: 75
New progress: 100/325 | successful: 100
New progress: 125/325 | successful: 125
New progress: 150/325 | successful: 150
New progress: 175/325 | successful: 175
New progress: 200/325 | successful: 200
New progress: 225/325 | successful: 225
New progress: 250/325 | successful: 250
New progress: 275/325 | successful: 275
New progress: 300/325 | successful: 300
New progress: 325/325 | successful: 325
New done: 325/325
Artifact zipped.
